# Prepare Models and other data for Deployment on Streamlit

Import models from ML workflow and export into app/ dir

## Set Up

Packages

In [ ]:
import sys, os

os.environ["OMP_NUM_THREADS"] = "1"  # for torch/sklearn MacOS conflict
sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
import src  # for pipelines
from app_config import CHOSEN_MODEL_DICT

import numpy as np
import pandas as pd
import joblib
from shutil import rmtree
import shap

## Imports/Exports

Initialize paths

In [ ]:
def clean_make_dir(directory):
    if directory.exists():
        rmtree(directory)
    directory.mkdir(exist_ok=False, parents=True)


## MODELS
cal_model_output_dir = BASE_PATH / "app" / "models"
clean_make_dir(cal_model_output_dir)

## PIPELINES
pipe_output_dir = BASE_PATH / "app" / "preprocessors"
clean_make_dir(pipe_output_dir)

## ALL PREDS
pred_output_dir = BASE_PATH / "app" / "all_preds"
clean_make_dir(pred_output_dir)

## BIN THRESHOLDS
threshold_output_dir = BASE_PATH / "app" / "bin_thresholds"
clean_make_dir(threshold_output_dir)

## SHAP explainers
shap_output_dir = BASE_PATH / "app" / "shap_explainers"
clean_make_dir(shap_output_dir)

In [ ]:
## GET OUTCOME PIPELINES, CHOSEN MODELS, AND ASSOCIATED PREDS/THRESHOLDS
for outcome_name, chosen_model in CHOSEN_MODEL_DICT.items():
    print(f"Outcome: {outcome_name}, model: {chosen_model}")
    # ================> Pipelines (shared by all models)
    pipeline = joblib.load(
        BASE_PATH / "data" / "pipelines2" / f"{outcome_name}_pipeline.joblib"
    )
    joblib.dump(pipeline, pipe_output_dir / f"{outcome_name}_pipeline.joblib")
    # ================> MODELS
    # Calibrated models for predictions
    cal_model = joblib.load(
        BASE_PATH / "models" / "calibrated" / outcome_name / f"{chosen_model}.joblib"
    )
    joblib.dump(
        cal_model, cal_model_output_dir / f"{outcome_name}_{chosen_model}.joblib"
    )
    # ================> All Preds
    test_preds = pd.read_parquet(
        BASE_PATH
        / "results"
        / "app"
        / "all_preds"
        / outcome_name
        / f"{chosen_model}.parquet"
    )
    test_preds.to_parquet(pred_output_dir / f"{outcome_name}_{chosen_model}.parquet")
    # ================> Bin Thresholds
    bin_thresholds = np.load(
        BASE_PATH
        / "results"
        / "app"
        / "bin_thresholds"
        / outcome_name
        / f"{chosen_model}.npz"
    )
    np.savez(
        threshold_output_dir / f"{outcome_name}_{chosen_model}.npz",
        thresholds=bin_thresholds["thresholds"],
    )
    # ================> SHAP explainer
    X_test = pd.read_parquet(
        BASE_PATH / "data" / "processed" / outcome_name / "X_test.parquet"
    )

    # No NN models used
    model_path = (
        BASE_PATH / "models" / "trained" / outcome_name / f"{chosen_model}.joblib"
    )
    un_cal_model = joblib.load(model_path)

    ## Hard-code tree explainer bc all chosen are either XGB or LGBM
    print("Fitting SHAP...")
    explainer = shap.TreeExplainer(
        model=un_cal_model,
        data=X_test,
        feature_perturbation="interventional",
        model_output="raw",  # faster computation than probability
        feature_names=X_test.columns.tolist(),
    )

    joblib.dump(explainer, shap_output_dir / f"{outcome_name}.joblib")

# Save feature names
feature_names = X_test.columns.tolist()
joblib.dump(feature_names, shap_output_dir / "feature_names.joblib")